In [1]:
# environment check
import sys, platform, torch, torchvision, sklearn, cv2, PIL
import numpy as np, pandas as pd, matplotlib, seaborn

print("Python:", sys.version.split()[0])
print("Platform:", platform.system())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("OpenCV:", cv2.__version__)

Python: 3.14.3
Platform: Windows
PyTorch: 2.14.1+cpu
Torchvision: 0.29.1+cpu
CUDA available: False
NumPy: 2.4.3
scikit-learn: 1.8.0
OpenCV: 5.0.0


In [2]:
# global paths 
from pathlib import Path
import random, os

PROJECT_ROOT = Path(r"C:\Users\Admin\lungprints")
DATA_DIR = PROJECT_ROOT / "data" / "chest_xray"
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
MODEL_DIR = RESULTS_DIR / "models"
LOG_DIR = RESULTS_DIR / "logs"

for p in [FIG_DIR, MODEL_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Data dir exists:", DATA_DIR.exists())
print("Device:", DEVICE)

Project root: C:\Users\Admin\lungprints
Data dir: C:\Users\Admin\lungprints\data\chest_xray
Data dir exists: False
Device: cpu


In [3]:
from pathlib import Path

DATA = Path(r"C:\Users\Admin\lungprints\data")

print("data exists:", DATA.exists())
print("\nTop-level contents:")
for p in DATA.iterdir():
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"  [{kind}] {p.name}")

data exists: True

Top-level contents:
  [DIR ] pneumonia


In [4]:
for p in DATA.iterdir():
    if p.is_dir():
        print("=" * 60)
        print("Folder:", p.name)
        for sub in p.iterdir():
            kind = "DIR " if sub.is_dir() else "FILE"
            print(f"  [{kind}] {sub.name}")

Folder: pneumonia
  [DIR ] chest_xray


In [5]:
from pathlib import Path

BASE = Path(r"C:\Users\Admin\lungprints\data")
REQUIRED = ["train", "val", "test"]

def has_splits(folder: Path) -> bool:
    return all((folder / s).is_dir() for s in REQUIRED)

def find_dataset_root(base: Path, depth=3, current=0):
    if current > depth:
        return None
    if has_splits(base):
        return base
    for sub in base.iterdir():
        if sub.is_dir():
            found = find_dataset_root(sub, depth, current + 1)
            if found:
                return found
    return None

DATA_DIR = find_dataset_root(BASE)
print("Detected DATA_DIR:", DATA_DIR)

if DATA_DIR:
    for split in REQUIRED:
        for cls in ["NORMAL", "PNEUMONIA"]:
            folder = DATA_DIR / split / cls
            n = len(list(folder.glob("*"))) if folder.exists() else 0
            print(f"{split:6s} | {cls:10s} | {n:5d}")

Detected DATA_DIR: C:\Users\Admin\lungprints\data\pneumonia\chest_xray
train  | NORMAL     |  1341


train  | PNEUMONIA  |  3875
val    | NORMAL     |     8
val    | PNEUMONIA  |     8
test   | NORMAL     |   234
test   | PNEUMONIA  |   390
